# 📊 Notebook 1 — Analyse exploratoire des données (EDA)
**Projet : Contrôle qualité des données Proxima**

Ce notebook documente l'analyse de départ effectuée avant l'écriture du pipeline.
Il permet de comprendre la structure, la qualité et les particularités des données sources.

---

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

sns.set_theme(style="whitegrid", palette="muted")
print("✅ Imports OK")

## 2. Chargement du fichier CSV

In [ ]:
# Adapter le chemin selon l'environnement
CSV_PATH = "data.csv"

df = pd.read_csv(CSV_PATH, sep=None, engine='python')
print(f"✅ Fichier chargé : {df.shape[0]} lignes × {df.shape[1]} colonnes")

## 3. Structure générale

In [ ]:
print("=== DIMENSIONS ===")
print(f"Lignes   : {df.shape[0]}")
print(f"Colonnes : {df.shape[1]}")
print()
print("=== TYPES DE DONNÉES ===")
dtype_counts = df.dtypes.value_counts()
for dtype, count in dtype_counts.items():
    print(f"  {dtype} : {count} colonne(s)")

In [ ]:
# Aperçu des premières lignes
df.head(5)

## 4. Répartition par marché

Le fichier CSV contient plusieurs sous-ensembles de données selon le champ `marche`.
Il est important de les identifier car ils peuvent avoir des structures différentes.

In [ ]:
marche_counts = df['marche'].value_counts(dropna=False)
print("=== RÉPARTITION PAR MARCHÉ ===")
print(marche_counts)

fig, ax = plt.subplots(figsize=(6, 3.5))
colors = ['#4285f4', '#34a853', '#ea4335', '#fa7b17']
marche_counts.plot(kind='bar', ax=ax, color=colors[:len(marche_counts)], edgecolor='white', linewidth=0.5)
ax.set_title("Répartition des lignes par marché", fontweight='bold', pad=12)
ax.set_xlabel("")
ax.set_ylabel("Nombre de lignes")
ax.tick_params(axis='x', rotation=0)
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Analyse du filtre contrats actifs

Deux colonnes permettent d'identifier les contrats actifs :
- `STOCK = 1` → site en portefeuille
- `DT_EFF_RESIL_CNT` vide → contrat non résilié

> ⚠️ La colonne `CDSITP` (statut définitif) n'est pas dans le CSV — elle vient du datamart.

In [ ]:
print("=== STOCK ===")
print(df['STOCK'].value_counts(dropna=False))
print()
print("=== DT_EFF_RESIL_CNT ===")
print(df['DT_EFF_RESIL_CNT'].value_counts(dropna=False))
print()

# Application du filtre
df_active = df[
    (df['STOCK'] == 1.0) &
    (df['DT_EFF_RESIL_CNT'].isna())
].copy()

print(f"=== RÉSULTAT DU FILTRE ===")
print(f"Avant filtre : {len(df)} lignes")
print(f"Après filtre : {len(df_active)} lignes actives")
print(f"Exclues      : {len(df) - len(df_active)} lignes")

## 6. Taux de remplissage des colonnes clés

On se concentre sur les 25 colonnes utilisées par le projet.

In [ ]:
KEY_COLS = [
    # Identification
    'STOCK', 'DT_EFF_RESIL_CNT', 'NU_CNT', 'PTGST', 'ID_SITE', 'NOM_CLI', 'NOM_SITE', 'marche',
    # Adresse
    'NUM_VOIE_SITE', 'RUE_SITE', 'NUM_RUE_SITE', 'LIEU_DIT_SITE',
    'CP_SITE', 'VILLE_SITE', 'PAYS_SITE',
    # GPS
    'COORD_X_SITE', 'COORD_Y_SITE', 'QUALITE_GEOCODAGE',
    # Techniques
    'SURFACE', 'SURFACE_DPDCE',
    'CAPITAUX_MURS_BATIMENTS', 'CAPITAUX_CONTENU',
    'CAPITAUX_MATERIEL', 'CAPITAUX_MARCHANDISES', 'CAPITAUX_TT'
]

fill_rates = []
for col in KEY_COLS:
    if col in df_active.columns:
        total = len(df_active)
        filled = df_active[col].notna().sum()
        rate = filled / total * 100 if total > 0 else 0
        fill_rates.append({'colonne': col, 'remplies': filled, 'total': total, 'taux_%': round(rate, 1)})

fill_df = pd.DataFrame(fill_rates).sort_values('taux_%', ascending=False)
fill_df

In [ ]:
# Visualisation
fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#34a853' if x >= 80 else '#fa7b17' if x >= 40 else '#ea4335' for x in fill_df['taux_%']]
bars = ax.barh(fill_df['colonne'], fill_df['taux_%'], color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlim(0, 110)
ax.axvline(80, color='#34a853', linestyle='--', linewidth=1, alpha=0.5, label='Seuil 80%')
ax.axvline(40, color='#fa7b17', linestyle='--', linewidth=1, alpha=0.5, label='Seuil 40%')
for bar, val in zip(bars, fill_df['taux_%']):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f'{val}%', va='center', fontsize=9)
ax.set_title("Taux de remplissage des colonnes clés (contrats actifs)", fontweight='bold', pad=12)
ax.set_xlabel("Taux de remplissage (%)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("\n🟢 >= 80% | 🟠 40-80% | 🔴 < 40%")

## 7. Analyse des champs adresse

In [ ]:
addr_cols = ['NUM_VOIE_SITE', 'RUE_SITE', 'NUM_RUE_SITE', 'LIEU_DIT_SITE', 'CP_SITE', 'VILLE_SITE', 'PAYS_SITE']

print("=== VALEURS MANQUANTES PAR CHAMP ADRESSE ===")
for col in addr_cols:
    if col in df_active.columns:
        missing = df_active[col].isna().sum()
        total = len(df_active)
        print(f"  {col:<22} : {missing}/{total} manquants ({missing/total*100:.1f}%)")

print()
print("=== PAYS_SITE — valeurs distinctes ===")
print(df_active['PAYS_SITE'].value_counts(dropna=False))

In [ ]:
# CP_SITE — problème du zéro initial
print("=== CP_SITE — exemples de valeurs brutes ===")
print(df_active['CP_SITE'].dropna().head(15).tolist())
print()
print("=> Stocké en float : le zéro initial est perdu (ex: 4100.0 = 04100)")
print()

# Correction
def fix_cp(val):
    try:
        return str(int(val)).zfill(5)
    except:
        return None

df_active['CP_SITE_FIXED'] = df_active['CP_SITE'].apply(fix_cp)
print("=== CP_SITE après correction ===")
print(df_active['CP_SITE_FIXED'].dropna().head(15).tolist())

In [ ]:
# Redondance RUE_SITE vs NUM_RUE_SITE
print("=== Redondance RUE_SITE vs NUM_RUE_SITE ===")
both_filled = df_active['RUE_SITE'].notna() & df_active['NUM_RUE_SITE'].notna()
only_rue    = df_active['RUE_SITE'].notna() & df_active['NUM_RUE_SITE'].isna()
only_num    = df_active['RUE_SITE'].isna()  & df_active['NUM_RUE_SITE'].notna()
both_empty  = df_active['RUE_SITE'].isna()  & df_active['NUM_RUE_SITE'].isna()

print(f"  Les deux renseignés    : {both_filled.sum()}")
print(f"  RUE_SITE seul          : {only_rue.sum()}")
print(f"  NUM_RUE_SITE seul      : {only_num.sum()}")
print(f"  Les deux vides         : {both_empty.sum()}")
print()
print("Exemples où les deux sont renseignés :")
df_active[both_filled][['RUE_SITE','NUM_RUE_SITE']].head(5)

## 8. Analyse des coordonnées GPS

In [ ]:
gps_cols = ['COORD_X_SITE', 'COORD_Y_SITE', 'QUALITE_GEOCODAGE']
print("=== REMPLISSAGE GPS ===")
for col in gps_cols:
    filled = df_active[col].notna().sum()
    print(f"  {col:<22} : {filled}/{len(df_active)} renseignés")

print()
# Cas où GPS est vide mais adresse présente (scénario A — valide)
has_address = df_active['CP_SITE'].notna() & df_active['VILLE_SITE'].notna()
no_gps      = df_active['COORD_X_SITE'].isna() | df_active['COORD_Y_SITE'].isna()
has_gps     = df_active['COORD_X_SITE'].notna() & df_active['COORD_Y_SITE'].notna()

print(f"Scénario A (adresse ✅, GPS absent) : {(has_address & no_gps).sum()} lignes → VALIDE")
print(f"Scénario B (adresse ✅, GPS ✅)     : {(has_address & has_gps).sum()} lignes → à vérifier cohérence")
print(f"Scénario C (adresse ❌, GPS ✅)     : {(~has_address & has_gps).sum()} lignes → à vérifier pays")
print(f"Cas dégénéré (adresse ❌, GPS ❌)   : {(~has_address & ~has_gps).sum()} lignes → anomalie grave")

## 9. Analyse des données techniques (capitaux & superficie)

In [ ]:
tech_cols = [
    'SURFACE', 'SURFACE_DPDCE',
    'CAPITAUX_MURS_BATIMENTS', 'CAPITAUX_CONTENU',
    'CAPITAUX_MATERIEL', 'CAPITAUX_MARCHANDISES', 'CAPITAUX_TT'
]

print("=== REMPLISSAGE DONNÉES TECHNIQUES ===")
for col in tech_cols:
    if col in df_active.columns:
        filled    = df_active[col].notna().sum()
        non_zero  = (df_active[col].fillna(0) > 0).sum()
        print(f"  {col:<35} : {filled}/{len(df_active)} renseignés, {non_zero} non nuls")

print()
# Règle 5 : au moins un champ technique renseigné et > 0
has_surface  = (df_active['SURFACE'].fillna(0) > 0) | (df_active['SURFACE_DPDCE'].fillna(0) > 0)
has_capitaux = (
    (df_active['CAPITAUX_MURS_BATIMENTS'].fillna(0) > 0) |
    (df_active['CAPITAUX_CONTENU'].fillna(0) > 0) |
    (df_active['CAPITAUX_MATERIEL'].fillna(0) > 0) |
    (df_active['CAPITAUX_MARCHANDISES'].fillna(0) > 0)
)
has_tech = has_surface | has_capitaux
missing_tech = ~has_tech

print(f"Lignes avec au moins 1 donnée technique : {has_tech.sum()}/{len(df_active)}")
print(f"Lignes SANS aucune donnée technique     : {missing_tech.sum()}/{len(df_active)} ← anomalie R5")

In [ ]:
# Distribution de CAPITAUX_CONTENU (seul capital renseigné dans l'échantillon)
cap = df_active['CAPITAUX_CONTENU'].dropna()
if len(cap) > 0:
    print("=== CAPITAUX_CONTENU — statistiques ===")
    print(cap.describe())
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.hist(cap, bins=15, color='#4285f4', edgecolor='white', linewidth=0.5)
    ax.set_title("Distribution de CAPITAUX_CONTENU", fontweight='bold')
    ax.set_xlabel("Valeur (€)")
    ax.set_ylabel("Fréquence")
    plt.tight_layout()
    plt.show()
else:
    print("CAPITAUX_CONTENU vide dans cet échantillon.")

## 10. Statut juridique (QUALITE_JURID)

In [ ]:
print("=== QUALITE_JURID — valeurs distinctes ===")
print(df_active['QUALITE_JURID'].value_counts(dropna=False))
print()
print("⚠️ Valeur 'L' semble non normalisée par rapport à 'LOC' — à clarifier avec Alain.")

## 11. Synthèse — Points clés identifiés

In [ ]:
print("""
SYNTHÈSE DE L'ANALYSE EXPLORATOIRE
====================================

📁 STRUCTURE
  - 154 colonnes au total, 25 utilisées par le projet
  - Le CSV mélange plusieurs marchés (PART, RI&GAR, NaN)
  - Les lignes NaN dans 'marche' semblent être des artefacts à exclure

🔎 FILTRE CONTRATS ACTIFS
  - STOCK = 1 : filtre disponible dans le CSV
  - CDSITP : absent du CSV, vient du datamart après merge
  - DT_EFF_RESIL_CNT : date de résiliation, vide = actif

📍 ADRESSE
  - CP_SITE stocké en float → conversion nécessaire (zéro initial)
  - RUE_SITE et NUM_RUE_SITE redondants → à arbitrer avec Alain
  - PAYS_SITE souvent vide, pourtant indispensable pour la validation
  - ADRESSE_SITE vide à 100% → inutilisable

🛰️ GPS
  - COORD_X_SITE et COORD_Y_SITE vides dans l'échantillon
  - Probable alimentation via le datamart
  - QUALITE_GEOCODAGE vide également

💰 DONNÉES TECHNIQUES
  - Seul CAPITAUX_CONTENU renseigné dans l'échantillon
  - Les autres capitaux probablement alimentés via le datamart
  - CAPITAUX_TT toujours à 0 → inutilisable comme indicateur

⚠️ QUESTIONS OUVERTES
  - Clé de jointure exacte CSV × Datamart (à confirmer avec Alain)
  - Référence officielle : RUE_SITE ou NUM_RUE_SITE ?
  - Valeur 'L' dans QUALITE_JURID : normalisée ou erreur de saisie ?
  - Pays autres que FRANCE dans les vraies données ?
""")